In [2]:
import sys
from collections import defaultdict
import csv
from datetime import datetime, time, timedelta
from math import floor, lcm
from causis_api.const import login
login.username = "jinqiao.xue"
login.password = "1101BX@causis"
from causis_api.const import *
from causis_api.data import *


获取每日可交易期权合约并提取相关信息

In [ ]:
date_str = "2025-11-10"
opt_chain = all_instruments('O', date_str)
opt_chain = opt_chain[opt_chain["Code"] == "159915"]
opt_chain["expire"] = [datetime.strptime(i, "%Y-%m-%d") for i in opt_chain["EndDate"]]
tmp = instruments(opt_chain["Symbol"])[["OptType", "StrikePrice",'MinTick','Multiplier']]
opt_chain["OptType"] = tmp["OptType"].tolist()
opt_chain["Strike"] = tmp["StrikePrice"].tolist()
opt_chain["MinTick"] = tmp["MinTick"].tolist()
opt_chain["Multiplier"] = tmp["Multiplier"].tolist()
opt_chain[['Symbol','Name','BeginDate','EndDate','OptType','Strike',"MinTick","Multiplier"]]
opt_chain

,Symbol,Code,Name,BeginDate,EndDate,Type,Exchange,MinTick,Multiplier,expire,OptType,Strike
163204,O.CN.SZSE.159915.90005531,159915,创业板ETF购2025年12月1.7,2025-04-24,2025-12-24,FUTURE,None,None,None,2025-12-24,Call,1.70
163205,O.CN.SZSE.159915.90005532,159915,创业板ETF购2025年12月1.75,2025-04-24,2025-12-24,FUTURE,None,None,None,2025-12-24,Call,1.75
163206,O.CN.SZSE.159915.90005533,159915,创业板ETF购2025年12月1.8,2025-04-24,2025-12-24,FUTURE,None,None,None,2025-12-24,Call,1.80
163207,O.CN.SZSE.159915.90005534,159915,创业板ETF购2025年12月1.85,2025-04-24,2025-12-24,FUTURE,None,None,None,2025-12-24,Call,1.85
163208,O.CN.SZSE.159915.90005535,159915,创业板ETF购2025年12月1.9,2025-04-24,2025-12-24,FUTURE,None,None,None,2025-12-24,Call,1.90
...,...,...,...,...,...,...,...,...,...,...,...,...
163513,O.CN.SZSE.159915.90006532,159915,创业板ETF沽2025年12月3.7,2025-10-30,2025-12-24,FUTURE,None,None,None,2025-12-24,Put,3.70
163514,O.CN.SZSE.159915.90006533,159915,创业板ETF购2026年3月3.7,2025-10-30,2026-03-25,FUTURE,None,None,None,2026-03-25,Call,3.70
163515,O.CN.SZSE.159915.90006534,159915,创业板ETF沽2026年3月3.7,2025-10-30,2026-03-25,FUTURE,None,None,None,2026-03-25,Put,3.70
163516,O.CN.SZSE.159915.90006535,159915,创业板ETF购2026年6月3.7,2025-10-30,2026-06-24,FUTURE,None,None,None,2026-06-24,Call,3.70


获取指定日期分钟级别指定期权合约数据

In [25]:
df = get_price(["O.CN.SZSE.159915.90006532","O.CN.SZSE.159915.90006533",'O.CN.SZSE.159915.90006534','O.CN.SZSE.159915.90006536'],start_date = "2025-01-01",end_date = "2025-12-23",frequency= "minute1")
df=df[['SYMBOL','CLOCK','OPEN','HIGH','LOW','CLOSE','VOLUME']]
df=df[df['SYMBOL']=="O.CN.SZSE.159915.90006532"]
df['DATE']=pd.to_datetime(df['CLOCK']).dt.date
df = df[df['DATE']==pd.to_datetime("2025-12-23").date()]
df

,SYMBOL,CLOCK,OPEN,HIGH,LOW,CLOSE,VOLUME,DATE
9360,O.CN.SZSE.159915.90006532,2025-12-23 09:31:00,0.5301,0.5301,0.5301,0.5301,0.0,2025-12-23
9361,O.CN.SZSE.159915.90006532,2025-12-23 09:32:00,0.5301,0.5301,0.5301,0.5301,0.0,2025-12-23
9362,O.CN.SZSE.159915.90006532,2025-12-23 09:33:00,0.5301,0.5301,0.5301,0.5301,0.0,2025-12-23
9363,O.CN.SZSE.159915.90006532,2025-12-23 09:34:00,0.5301,0.5301,0.5301,0.5301,0.0,2025-12-23
9364,O.CN.SZSE.159915.90006532,2025-12-23 09:35:00,0.5301,0.5301,0.5301,0.5301,0.0,2025-12-23
...,...,...,...,...,...,...,...,...
9595,O.CN.SZSE.159915.90006532,2025-12-23 14:56:00,0.5231,0.5231,0.5231,0.5231,0.0,2025-12-23
9596,O.CN.SZSE.159915.90006532,2025-12-23 14:57:00,0.5231,0.5231,0.5231,0.5231,0.0,2025-12-23
9597,O.CN.SZSE.159915.90006532,2025-12-23 14:58:00,0.5231,0.5231,0.5231,0.5231,0.0,2025-12-23
9598,O.CN.SZSE.159915.90006532,2025-12-23 14:59:00,0.5231,0.5231,0.5231,0.5231,0.0,2025-12-23


获取ETF分钟级别数据

In [18]:
df = get_price("S.CN.SZSE.159915",start_date = "2025-01-01",end_date = "2026-01-01",frequency= "minute1")
df[['SYMBOL','CLOCK','OPEN','HIGH','LOW','CLOSE','VOLUME']]

,SYMBOL,CLOCK,OPEN,HIGH,LOW,CLOSE,VOLUME
0,S.CN.SZSE.159915,2025-01-02 09:31:00,2.096,2.097,2.085,2.086,69447600.0
1,S.CN.SZSE.159915,2025-01-02 09:32:00,2.086,2.087,2.083,2.083,32664400.0
2,S.CN.SZSE.159915,2025-01-02 09:33:00,2.083,2.085,2.080,2.080,27050900.0
3,S.CN.SZSE.159915,2025-01-02 09:34:00,2.081,2.081,2.074,2.076,38561000.0
4,S.CN.SZSE.159915,2025-01-02 09:35:00,2.076,2.080,2.075,2.076,27966200.0
...,...,...,...,...,...,...,...
58315,S.CN.SZSE.159915,2025-12-31 14:56:00,3.192,3.192,3.189,3.190,9477600.0
58316,S.CN.SZSE.159915,2025-12-31 14:57:00,3.190,3.191,3.188,3.190,6745000.0
58317,S.CN.SZSE.159915,2025-12-31 14:58:00,3.190,3.190,3.190,3.190,268000.0
58318,S.CN.SZSE.159915,2025-12-31 14:59:00,3.190,3.190,3.190,3.190,0.0


下一个交易日


In [24]:
date = "2025-12-20"
result = get_next_trading_date(date, 1)
result

'2025-12-22'

获取起点和终点之间的交易日

In [25]:
result = get_trading_dates("2025-07-10", "2025-07-20")
print(result)

['2025-07-10', '2025-07-11', '2025-07-14', '2025-07-15', '2025-07-16', '2025-07-17', '2025-07-18']
